In [ ]:
from os import listdir as ls
import pandas as pd
import seaborn as sns
import pycountry
from matplotlib.pyplot import subplots
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.utils import get_country_short_name
from emu_renewal.plotting import get_avail_groupings
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios

set_matplotlib_formats("svg")

https://unstats.un.org/unsd/methodology/m49/overview/#

In [ ]:
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)
mob_source = "fb_singletile_mob"

In [ ]:
# Get the dispersion ratios
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)
ratio_vals = get_median_ratios(ratio_dists, mob_source)
ratios = {get_country_short_name(k): v for k, v in ratio_vals.items()}

In [ ]:
# Get the mob_exp values
idatas, _ = get_idatas_for_mob_type(job_path, all_countries, mob_source)
mob_exp_df = pd.DataFrame(columns=idatas)
for iso3 in idatas:
    mob_exp_df[iso3] = idatas[iso3].posterior["mob_exp"].to_dataframe()
grouping = get_avail_groupings(mob_exp_df.columns)

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.pyplot import get_cmap
from matplotlib.cm import ScalarMappable
from emu_renewal.inputs import get_gdps, get_country_pop
from emu_renewal.constants import MOB_SOURCE_COLOURS

In [ ]:
comb_df = pd.DataFrame()
comb_df["mob_exp"] = mob_exp_df.median()
comb_df["ratio"] = pd.Series(ratio_vals)
comb_df["gdp"] = get_gdps(2020)
comb_df["pop"] = {c: get_country_pop(c) / 1e6 for c in all_countries}


In [ ]:
sns.scatterplot(x="ratio", y="mob_exp", hue="gdp", size="pop", data=comb_df, sizes=(25, 500))

In [ ]:
plt.scatter(comb_df["ratio"], comb_df["mob_exp"])
plt.xlabel("dispersion ratio")
plt.ylabel("mobility exponent")

In [ ]:
fig, axes = subplots(3, 5, figsize=[12, 12], sharey=True)
flat_axes = axes.ravel()

norm = Normalize(vmin=min(ratios.values()), vmax=max(ratios.values()))
cmap = get_cmap(f"{MOB_SOURCE_COLOURS[mob_source].capitalize()}s_r")
palette = {c: cmap(norm(v)) for c, v in ratios.items()}

for r, (region, countries) in enumerate(grouping.items()):
    ax = flat_axes[r]
    ax.set_title(region)
    plot_df = mob_exp_df[countries]
    plot_df.columns = plot_df.columns.map(get_country_short_name)
    sns.violinplot(plot_df, ax=ax, palette=palette)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=90)

ax = flat_axes[r + 1]
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, location="left")
ax.remove()

fig.tight_layout()